<a href="https://colab.research.google.com/github/obok127/Data-Science/blob/main/NDCG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# NDCG stands for Normalized Discounted Cumulative Gain.
# ============================================================
# NDCG tutorial with public data (MovieLens small)
# Goal:
# - Use a public dataset
# - Build a simple ranking score
# - Compute DCG / NDCG from scratch
# - Compare with random ranking
# ============================================================
import io
import zipfile
from urllib.request import urlopen

import numpy as np
import pandas as pd
from sklearn.metrics import ndcg_score

# ------------------------------------------------------------
# 1. Download public MovieLens data
# ------------------------------------------------------------
url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"

with urlopen(url) as response:
    zip_content = response.read()

with zipfile.ZipFile(io.BytesIO(zip_content)) as z:
    ratings = pd.read_csv(z.open("ml-latest-small/ratings.csv"))
    movies = pd.read_csv(z.open("ml-latest-small/movies.csv"))

print("ratings shape:", ratings.shape)
print("movies shape:", movies.shape)
print(ratings.head())


ratings shape: (100836, 4)
movies shape: (9742, 3)
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931


In [2]:
# ------------------------------------------------------------
# 2. Keep users with enough ratings
#    We want enough history to split into train/test
# ------------------------------------------------------------
user_counts = ratings["userId"].value_counts()
active_users = user_counts[user_counts >= 20].index

ratings = ratings[ratings["userId"].isin(active_users)].copy()
ratings = ratings.sort_values(["userId", "timestamp"]).reset_index(drop=True)

print("Active users:", ratings["userId"].nunique())
print("Filtered ratings:", ratings.shape)

Active users: 610
Filtered ratings: (100836, 4)


In [3]:
# ------------------------------------------------------------
# 3. Train/test split per user
#    For each user:
#    - training = earlier ratings
#    - test = latest 5 ratings
#
# This is still a simplified educational setup,
# but much better than using all ratings at once.
# ------------------------------------------------------------
train_parts = []
test_parts = []

for user_id, group in ratings.groupby("userId"):
    group = group.sort_values("timestamp")

    if len(group) < 10:
        continue

    test_part = group.tail(5)
    train_part = group.iloc[:-5]

    train_parts.append(train_part)
    test_parts.append(test_part)

train = pd.concat(train_parts).reset_index(drop=True)
test = pd.concat(test_parts).reset_index(drop=True)

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (97786, 4)
Test shape: (3050, 4)


In [4]:
# ------------------------------------------------------------
# 4. Build a very simple ranking model
#
# We'll predict how good a movie is using its average rating
# from the TRAIN data only.
#
# To make it a bit more stable, we use a smoothed mean:
# score = (movie_mean * n + global_mean * m) / (n + m)
#
# where:
# - n = number of ratings for that movie
# - m = smoothing strength (hyperparameter)
# ------------------------------------------------------------
global_mean = train["rating"].mean()
m = 10  # smoothing strength

movie_stats = (
    train.groupby("movieId")
    .agg(movie_mean=("rating", "mean"),
         n_ratings=("rating", "size"))
    .reset_index()
)

movie_stats["pred_score"] = (
    (movie_stats["movie_mean"] * movie_stats["n_ratings"] + global_mean * m)
    / (movie_stats["n_ratings"] + m)
)

movie_stats.head()

,movieId,movie_mean,n_ratings,pred_score
0,1,3.910798,213,3.892238
1,2,3.425234,107,3.431360
2,3,3.218750,48,3.266709
3,4,2.250000,6,3.029320
4,5,3.074468,47,3.148581


In [5]:
# ------------------------------------------------------------
# 5. Attach predicted scores to test data
#    If a movie is unseen in training, fall back to global mean
# ------------------------------------------------------------
test_scored = test.merge(
    movie_stats[["movieId", "pred_score"]],
    on="movieId",
    how="left"
)

test_scored["pred_score"] = test_scored["pred_score"].fillna(global_mean)

test_scored = test_scored.merge(
    movies[["movieId", "title"]],
    on="movieId",
    how="left"
)

test_scored.head()

,userId,movieId,rating,timestamp,pred_score,title
0,1,1445,3.0,964984112,2.935570,McHale's Navy (1997)
1,1,553,5.0,964984153,3.767668,Tombstone (1993)
2,1,2478,4.0,964984169,3.213403,¡Three Amigos! (1986)
3,1,2012,4.0,964984176,3.368096,Back to the Future Part III (1990)
4,1,2492,4.0,965719662,3.197941,20 Dates (1998)


In [6]:
# ------------------------------------------------------------
# 6. Define DCG and NDCG from scratch
# ------------------------------------------------------------
def dcg_at_k(relevances, k=5):
    """
    relevances: array-like of true relevance values
                already arranged in the model's ranked order
    """
    relevances = np.asarray(relevances)[:k]
    if len(relevances) == 0:
        return 0.0

    positions = np.arange(1, len(relevances) + 1)
    discounts = np.log2(positions + 1)

    # Common DCG formula
    gains = (2 ** relevances - 1) / discounts
    return np.sum(gains)

def ndcg_at_k(y_true, y_score, k=5):
    """
    y_true: true relevance values
    y_score: model predicted scores
    """
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    # order predicted by model
    pred_order = np.argsort(y_score)[::-1]
    ranked_true = y_true[pred_order]

    # ideal order
    ideal_order = np.argsort(y_true)[::-1]
    ideal_true = y_true[ideal_order]

    dcg = dcg_at_k(ranked_true, k=k)
    idcg = dcg_at_k(ideal_true, k=k)

    if idcg == 0:
        return 0.0
    return dcg / idcg

In [7]:
# ------------------------------------------------------------
# 7. Evaluate NDCG@5 for each user
#    Here:
#    - true relevance = user's actual rating in the test set
#    - predicted score = our simple movie score
#
# This means:
# "Did our model rank the movies the user ended up liking
#  more near the top?"
# ------------------------------------------------------------
user_results = []

rng = np.random.default_rng(42)

for user_id, group in test_scored.groupby("userId"):
    if len(group) < 2:
        continue

    y_true = group["rating"].to_numpy()
    y_score_model = group["pred_score"].to_numpy()
    y_score_random = rng.random(len(group))

    ndcg_model = ndcg_at_k(y_true, y_score_model, k=5)
    ndcg_random = ndcg_at_k(y_true, y_score_random, k=5)

    # sklearn check
    ndcg_model_sklearn = ndcg_score([y_true], [y_score_model], k=5)

    user_results.append({
        "userId": user_id,
        "n_test_items": len(group),
        "ndcg_model": ndcg_model,
        "ndcg_model_sklearn": ndcg_model_sklearn,
        "ndcg_random": ndcg_random
    })

results_df = pd.DataFrame(user_results)

print(results_df.head())
print()
print("Average NDCG@5 (model):  ", round(results_df["ndcg_model"].mean(), 4))
print("Average NDCG@5 (random): ", round(results_df["ndcg_random"].mean(), 4))
print("Average sklearn NDCG@5:  ", round(results_df["ndcg_model_sklearn"].mean(), 4))

   userId  n_test_items  ndcg_model  ndcg_model_sklearn  ndcg_random
0       1             5    1.000000            1.000000     0.806382
1       2             5    0.908651            0.973306     1.000000
2       3             5    1.000000            1.000000     1.000000
3       4             5    0.985284            0.993684     0.985284
4       5             5    1.000000            1.000000     0.924648

Average NDCG@5 (model):   0.9119
Average NDCG@5 (random):  0.8772
Average sklearn NDCG@5:   0.9609


In [8]:
# ------------------------------------------------------------
# 8. Inspect one user in detail
#    This is the best part for understanding.
# ------------------------------------------------------------
sample_user = results_df.sort_values("ndcg_model", ascending=False).iloc[0]["userId"]
sample_user = int(sample_user)

sample = test_scored[test_scored["userId"] == sample_user].copy()

# model ranking
sample_pred = sample.sort_values("pred_score", ascending=False)[
    ["title", "rating", "pred_score"]
].reset_index(drop=True)

# ideal ranking
sample_ideal = sample.sort_values("rating", ascending=False)[
    ["title", "rating", "pred_score"]
].reset_index(drop=True)

print(f"Sample user: {sample_user}")
print("\n[Model ranking]")
print(sample_pred)

print("\n[Ideal ranking based on true relevance]")
print(sample_ideal)

y_true = sample["rating"].to_numpy()
y_score = sample["pred_score"].to_numpy()

print("\nDCG@5:", round(dcg_at_k(sample_pred["rating"].to_numpy(), k=5), 4))
print("NDCG@5:", round(ndcg_at_k(y_true, y_score, k=5), 4))

Sample user: 3

[Model ranking]
                        title  rating  pred_score
0     Schindler's List (1993)     0.5    4.191640
1  Requiem for a Dream (2000)     0.5    3.903549
2        Rescuers, The (1977)     0.5    3.453142
3      You've Got Mail (1998)     0.5    3.266709
4            Snow Dogs (2002)     0.5    3.231274

[Ideal ranking based on true relevance]
                        title  rating  pred_score
0  Requiem for a Dream (2000)     0.5    3.903549
1        Rescuers, The (1977)     0.5    3.453142
2     Schindler's List (1993)     0.5    4.191640
3            Snow Dogs (2002)     0.5    3.231274
4      You've Got Mail (1998)     0.5    3.266709

DCG@5: 1.2213
NDCG@5: 1.0


In [9]:
# ------------------------------------------------------------
# 9. Optional: tiny hand-made example for intuition
# ------------------------------------------------------------
true_relevance = np.array([5, 4, 1, 3, 2])   # actual usefulness
pred_scores    = np.array([0.9, 0.8, 0.7, 0.4, 0.3])  # model scores

print("Hand-made example NDCG@5:", round(ndcg_at_k(true_relevance, pred_scores, k=5), 4))
print("Hand-made example sklearn:", round(ndcg_score([true_relevance], [pred_scores], k=5), 4))

Hand-made example NDCG@5: 0.989
Hand-made example sklearn: 0.9822
